# 01 EDA & Feature Engineering: College Football Attendance


In [9]:
import sys
print(sys.executable)

/opt/homebrew/opt/python@3.11/bin/python3.11


In [10]:
import sys
!{sys.executable} -m pip install python-dotenv requests pandas


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [11]:
!pip install python-dotenv requests pandas



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [12]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env in the project root

CFBD_API_KEY = os.getenv("CFBD_API_KEY")

if not CFBD_API_KEY:
    raise RuntimeError(
        "CFBD_API_KEY not found. Copy .env.example to .env and add your key "
        "(get one at https://collegefootballdata.com/key)."
    )


## Fetch data from the CFBD API

Pulling the `/games` endpoint for a given season as the first slice of data,
this is what attendance figures hang off of. Swap `YEAR` / add `week` or
`team` params as the analysis needs more granularity.

In [13]:
# Call the CFBD API using the key loaded from .env
import requests
import pandas as pd

YEAR = 2023  

BASE_URL = "https://api.collegefootballdata.com"
HEADERS = {
    "Authorization": f"Bearer {CFBD_API_KEY}",
    "Accept": "application/json",
}

response = requests.get(
    f"{BASE_URL}/games",
    headers=HEADERS,
    params={"year": YEAR, "seasonType": "regular"},
    timeout=30,
)
response.raise_for_status()

games_raw = response.json()
df = pd.DataFrame(games_raw)

print(f"Fetched {len(df)} rows for the {YEAR} regular season.")


Fetched 3595 rows for the 2023 regular season.


## Verify the fetch

In [14]:
#Confirm the fetch worked
df.head()

,id,season,week,seasonType,startDate,startTimeTBD,completed,neutralSite,conferenceGame,attendance,...,awayConference,awayPoints,awayLineScores,awayPostgameWinProbability,awayPregameElo,awayPostgameElo,excitementIndex,highlights,notes,playoff
0,401525434,2023,1,regular,2023-08-26T18:30:00.000Z,False,True,True,False,49000.0,...,American Athletic,3.0,"[0, 0, 0, 3]",0.001042,1471.0,1385.0,1.346908,,NaN,None
1,401540199,2023,1,regular,2023-08-26T19:30:00.000Z,False,True,True,False,NaN,...,UAC,7.0,"[7, 0, 0, 0]",0.025849,NaN,NaN,6.896909,,NaN,None
2,401520145,2023,1,regular,2023-08-26T21:30:00.000Z,False,True,False,True,17982.0,...,Conference USA,14.0,"[0, 7, 0, 7]",0.591999,1369.0,1370.0,6.821333,,NaN,None
3,401525450,2023,1,regular,2023-08-26T23:00:00.000Z,False,True,False,False,15356.0,...,FBS Independents,41.0,"[7, 3, 3, 28]",0.760751,1074.0,1122.0,5.311493,,NaN,None
4,401540628,2023,1,regular,2023-08-26T23:00:00.000Z,False,True,False,False,NaN,...,Patriot,13.0,"[3, 3, 7, 0]",0.077483,NaN,NaN,5.608758,,NaN,None


In [15]:
print("Shape:", df.shape)
df.info()

Shape: (3595, 34)
<class 'pandas.DataFrame'>
RangeIndex: 3595 entries, 0 to 3594
Data columns (total 34 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   id                          3595 non-null   int64  
 1   season                      3595 non-null   int64  
 2   week                        3595 non-null   int64  
 3   seasonType                  3595 non-null   str    
 4   startDate                   3595 non-null   str    
 5   startTimeTBD                3595 non-null   bool   
 6   completed                   3595 non-null   bool   
 7   neutralSite                 3595 non-null   bool   
 8   conferenceGame              3595 non-null   bool   
 9   attendance                  851 non-null    float64
 10  venueId                     3585 non-null   float64
 11  venue                       3585 non-null   str    
 12  homeId                      3595 non-null   int64  
 13  homeTeam                  

## Export to CSV so the team doesn't need their own API key



In [16]:
#Export the fetched data so teammates can work from a static file 

os.makedirs("../data/raw", exist_ok=True)
output_path = f"../data/raw/cfbd_games_{YEAR}.csv"
df.to_csv(output_path, index=False)

print(f"Saved to {output_path}")




Saved to ../data/raw/cfbd_games_2023.csv
